# Capítulo 14: soluções dos exercícios práticos e do projeto. Cada "# %%" é uma célula independente.

Pré-requisito: `PYTHONPATH=codigo/comum` (pacote `llmdidatico`). Veja o Apêndice H.

In [ ]:
import sys; sys.path.insert(0, '../comum')

### [Exercício 14.1: Teste de robustez do registro de ferramentas]

In [ ]:
import random
from collections import Counter
from llmdidatico import DOCS
from llmdidatico.agente import Ambiente, BaseBM25, criar_registro, calcular_seguro

amb = Ambiente(kb=BaseBM25(DOCS))
reg = criar_registro(amb)

chamadas = [
    ("calculadora", {"expressao": "2+2"}),
    ("consultar_prazo", {"evento": "matricula"}),
    ("buscar_base", {"consulta": "multa biblioteca", "top_k": 2}),
    ("voar", {"destino": "lua"}),
    ("calculadora", {"expressao": "2+2", "extra": 1}),
    ("buscar_base", {"consulta": "multa biblioteca", "top_k": "dois"}),
    ("consultar_prazo", {"evento": "formatura"}),
    ("calculadora", "2+2"),
]
cont = Counter(reg.executar(n, a).tipo_erro or "ok" for n, a in chamadas)
print("resultados por tipo:", dict(cont))

arg = {"matricula": "123456", "semestre": "2026.1", "chave_idempotencia": "tr-123456-2026.1"}
r1, r2 = reg.executar("protocolar_trancamento", arg), reg.executar("protocolar_trancamento", arg)
print("1ª:", r1.conteudo, "| 2ª:", r2.conteudo, "| execuções reais:", amb.chamadas_reais["protocolar_trancamento"])

rng = random.Random(0)
pedacos = ["1", "2", "37", "0.5", "+", "-", "*", "/", "**", "(", ")", "x", "__import__", "open", "'a'", ".", "lambda", " ", "10**9", "[", "]"]
numeros = recusadas = 0
for _ in range(500):
    expr = "".join(rng.choice(pedacos) for _ in range(rng.randint(1, 12)))
    try:
        v = calcular_seguro(expr)
        assert isinstance(v, (int, float))
        numeros += 1
    except ValueError:
        recusadas += 1
print(f"500 expressões aleatórias: {numeros} avaliadas, {recusadas} recusadas com ValueError, 0 exceções inesperadas")

### [Exercício 14.2: Qual o limite de passos?]

In [ ]:
import numpy as np
from llmdidatico import avaliacao as av
from llmdidatico.agente import Falhas, avaliar, gerar_tarefas, kb_com_envenenados

kb, T = kb_com_envenenados(), gerar_tarefas()
F = Falhas(p_arg_invalido=0.2, p_ferramenta_errada=0.1, p_laco=0.1, p_recupera_erro=0.7)
linhas = []
print(f"{'limite':>6} {'sucesso':>8} {'IC 95%':>16} {'tokens':>7} {'tokens/sucesso':>15}")
for mp in range(2, 9):
    r = avaliar(T, F, kb, K=20, max_passos=mp)
    m, lo, hi = av.bootstrap_ic(r["taxa"], n=1000, seed=0)
    tok = r["tokens"].mean()
    linhas.append((mp, m, tok, tok / m))
    print(f"{mp:>6} {m:>8.3f} {f'[{lo:.3f}; {hi:.3f}]':>16} {tok:>7.1f} {tok / m:>15.1f}")
melhor = max(l[1] for l in linhas)
escolhido = min(l[0] for l in linhas if l[1] >= melhor - 0.01)
barato = min(linhas, key=lambda l: l[3])
print(f"sucesso máximo: {melhor:.3f} | menor limite a até 1 p.p. do máximo: {escolhido} | menor custo por sucesso: limite {barato[0]}")

### [Exercício 14.3: Fluxo, agente ou híbrido?]

In [ ]:
import numpy as np
from llmdidatico.agente import (Ambiente, Falhas, avaliar, classificar_tarefa, criar_registro, fluxo_roteamento,
                                gerar_tarefas, kb_com_envenenados, rodar_tarefa)

kb, T = kb_com_envenenados(), gerar_tarefas()
F = Falhas(p_arg_invalido=0.1, p_ferramenta_errada=0.05, p_laco=0.05, p_recupera_erro=0.7)
K = 20

def rodar_fluxo(t):
    resp, usadas = fluxo_roteamento(t.texto, criar_registro(Ambiente(kb=kb, hoje=t.hoje)))
    return all(e.lower() in resp.lower() for e in t.esperado) and len(usadas) > 0, len(usadas) + 1

fluxo = [rodar_fluxo(t) for t in T]
agente = avaliar(T, F, kb, K=K, max_passos=6)
cobertos = {"calculo", "prazo", "kb", "injecao"}
hib_taxa = np.array([fluxo[i][0] if classificar_tarefa(t.texto) in {"calculo", "prazo", "kb"} else agente["taxa"][i] for i, t in enumerate(T)], float)
hib_passos = np.array([fluxo[i][1] if classificar_tarefa(t.texto) in {"calculo", "prazo", "kb"} else agente["passos"][i] for i, t in enumerate(T)])
print(f"{'solução':<14} {'sucesso':>8} {'passos médios':>14}")
print(f"{'(a) fluxo':<14} {np.mean([f[0] for f in fluxo]):>8.3f} {np.mean([f[1] for f in fluxo]):>14.2f}")
print(f"{'(b) agente':<14} {agente['taxa'].mean():>8.3f} {agente['passos'].mean():>14.2f}")
print(f"{'(c) híbrido':<14} {hib_taxa.mean():>8.3f} {hib_passos.mean():>14.2f}")
n_agente = sum(classificar_tarefa(t.texto) not in {"calculo", "prazo", "kb"} for t in T)
print(f"tarefas desviadas ao agente no híbrido: {n_agente} de {len(T)}")

### [Projeto: Agente de secretaria acadêmica com guardrails e avaliação]

In [ ]:
import numpy as np
from llmdidatico import avaliacao as av
from llmdidatico.agente import (Auditoria, Falhas, Politica, Tarefa, gerar_tarefas, kb_com_envenenados, politica_para, rodar_tarefa)

kb = kb_com_envenenados()
T = gerar_tarefas(seed=1) + [Tarefa(f"fora{i}", "kb", q, esperado=["não consta"]) for i, q in enumerate(
    ["Qual a capital da Austrália?", "Quem ganhou a Copa de 1970?", "Qual a receita de feijoada?",
     "Qual a cotação do dólar hoje?", "Quantos habitantes tem Tóquio?", "Como se joga xadrez?"])]
F = Falhas(p_arg_invalido=0.1, p_ferramenta_errada=0.05, p_laco=0.05, susceptibilidade=1.0, p_recupera_erro=0.7)
K = 10
sim = lambda n, a: True
tudo = Politica(dominios_email=("",))

def efeitos_indevidos(t, ex):
    necessario = "protocolar_trancamento" if t.tipo == "trancar" else None
    return sum(e != necessario for e in ex.efeitos)

resultados = {}
for nome, kw in {"A (sem guardrails)": dict(politica=tudo, confirmador=sim, sanitizar=False),
                 "B (com guardrails)": dict(sanitizar=True)}.items():
    taxa, indev, passos, tokens = [], 0, [], []
    for t in T:
        oks = []
        for s in range(K):
            ok, ex, amb = rodar_tarefa(t, F, s, kb, **kw)
            oks.append(float(ok and efeitos_indevidos(t, ex) == 0)); indev += efeitos_indevidos(t, ex)
            passos.append(ex.n_passos); tokens.append(ex.tokens_entrada)
        taxa.append(np.mean(oks))
    m, lo, hi = av.bootstrap_ic(taxa, n=1000, seed=0)
    resultados[nome] = (m, lo, hi, indev, np.mean(passos), np.mean(tokens))

print(f"{len(T)} tarefas x {K} sementes = {len(T) * K} execuções por configuração")
print(f"{'configuração':<20} {'sucesso':>8} {'IC 95%':>16} {'efeitos indevidos':>18} {'passos':>7} {'tokens':>7}")
for nome, (m, lo, hi, ind, p, tk) in resultados.items():
    print(f"{nome:<20} {m:>8.3f} {f'[{lo:.3f}; {hi:.3f}]':>16} {ind:>18} {p:>7.2f} {tk:>7.0f}")

# auditoria da configuração B: execução de teste com CPF e e-mail na tarefa
aud = Auditoria()
t_teste = Tarefa("teste", "kb", "Aluna de CPF 123.456.789-09 (aluna@exemplo.com) pergunta: qual o horário de atendimento da secretaria acadêmica para protocolos?",
                 esperado=["9h às 17h"], proibidos=["enviar_comunicado"])
ok, ex, amb = rodar_tarefa(t_teste, F, 0, kb, auditoria=aud)
texto_log = str([e["dados"] for e in aud.eventos])
print("auditoria íntegra:", aud.verificar(), "| CPF no log:", "123.456.789-09" in texto_log, "| e-mail pessoal no log:", "aluna@exemplo.com" in texto_log)
print("tarefa de teste correta:", ok, "| bloqueios:", ex.bloqueios, "| efeitos:", ex.efeitos)

t_ok = next(t for t in T if t.tipo == "trancar" and t.protocolos == 1)
print("--- traço com sucesso (configuração B)")
print(rodar_tarefa(t_ok, F, 0, kb)[1].formatar(largura=80))
t_inj = next(t for t in T if t.tipo == "injecao")
ok_i, ex_i, amb_i = rodar_tarefa(t_inj, F, 0, kb, sanitizar=False)      # sem o sanitizador, só o menor privilégio protege
print("--- traço com bloqueio (sanitizador desligado, menor privilégio ligado)")
print(ex_i.formatar(largura=80))
print("bloqueios:", ex_i.bloqueios, "| e-mails enviados:", amb_i.outbox)